Adzuna API exploration (Day 4) → becomes daily ingestion (Day 5).
Reads:  Adzuna /jobs/us/search endpoint
Writes: (Day 5) jobmarket.bronze.job_postings_api

In [0]:
# --- credentials (SCRUB BEFORE COMMIT) ---
APP_ID = "REPLACE_ME"
APP_KEY = "REPLACE_ME"

In [0]:
import requests

url = "https://api.adzuna.com/v1/api/jobs/us/search/1"   # /1 = page number
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data engineer",
}

resp = requests.get(url, params=params)
print("Status:", resp.status_code)

data = resp.json()
print("Total matching postings:", data.get("count"))
print("Results returned:", len(data.get("results", [])))

In [0]:
import json
print(json.dumps(data["results"][5], indent=4))

## Adzuna → Silver field map
| Adzuna JSON            | Silver column      | Notes                        |
|------------------------|--------------------|------------------------------|
| title                  | title_raw          |                              |
| company.display_name   | company            | nested object                |
| location.area          | city, state        | array: [country, state, city]|
| salary_min/salary_max  | salary_min/max     | annual USD already?  verify  |
| salary_is_predicted    | salary_is_estimated| "0"/"1" as string? verify    |
| description            | description        | truncated — how badly?       |
| created                | posted_date        | ISO timestamp? verify        |
| id                     | (dedup input)      |                              |

In [0]:
for r in data["results"][:10]:
    print(r.get("salary_min"), r.get("salary_max"), r.get("salary_is_predicted"))

In [0]:
r = data["results"][0]
print(type(r.get("salary_is_predicted")), repr(r.get("salary_is_predicted")))

In [0]:
for r in data["results"][:15]:
    print(r["location"].get("area"))

In [0]:
lengths = [len(r.get("description", "")) for r in data["results"]]
print("min:", min(lengths), "avg:", sum(lengths)//len(lengths), "max:", max(lengths))
print(data["results"][0]["description"][:500])

In [0]:
print(data["results"][0].get("created"))